<a href="https://colab.research.google.com/github/aathifsk1-gh/flyrank-assignment/blob/main/work/notebooks/w05_model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-08 — Capstone Modeling Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/aathifsk1-gh/flyrank-assignment01/blob/main/work/notebooks/w05_model.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [5]:
# --- Setup ---
import os, sys, subprocess

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/flyrank-bih/flyrank-ml-internship-starter"
REPO_DIR = "flyrank-ml-internship-starter"

if IN_COLAB:
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    os.chdir(REPO_DIR)
else:
    while not os.path.isdir("data/raw") and os.getcwd() != "/":
        os.chdir("..")

print("Working dir:", os.getcwd())
assert os.path.exists("data/raw/content_refresh_anonymized.csv"), "starter CSV not found"
print("Starter data found.")

Working dir: /content/flyrank-ml-internship-starter/flyrank-ml-internship-starter
Starter data found.


## 1. Method choice and why

*Which method from the toolkit, and why it fits your lane.*

Methods: Logistic Regression (readable baseline model) and Random Forest (stronger non-linear model).

**Why they fit this lane: **

We need a score for ranking (“which pages first?”), not only a yes/no.

Both models output probabilities we can rank by. Logistic regression is easy to explain; random forest usually captures interactions (volume × position × age) better. We compare both to the hand-written baseline on the same split and same metric (Precision@50).

We deliberately skip deep models — the gain here should come from combining observed signals honestly, not from opaque capacity.

In [6]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 2. Split design

*Grouped by client? Time-aware? Say why this split is honest for your question.*

Split: client-holdout (~20% of clients held out, not random rows).

**Why this is honest:**

Pages from the same client share style, niche, and traffic patterns. A random row split would leak that character into both train and test and inflate scores. Holding out whole clients asks: “does this work on a client the model never saw?” — closer to real use.

Seed fixed at 42 for reproducibility.

In [7]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.pipeline import Pipeline

RANDOM_STATE = 42

df = pd.read_csv("data/raw/content_refresh_anonymized.csv").copy()
df["is_declining_label"] = (df["trend_direction"] == "down").astype(int)

# Features — NO trend_direction / trend_pct (label leakage)
numeric_feats = [
    "impressions_90d", "clicks_90d", "sessions_90d", "pageviews_90d",
    "users_90d", "engaged_sessions_90d", "ai_sessions_90d", "scroll_events_90d",
    "ctr", "avg_position", "engagement_rate", "scroll_rate", "ai_traffic_pct",
    "word_count", "content_age_days", "days_since_last_update",
    "search_volume", "competition", "cpc",
]
cat_feats = ["content_type", "main_intent", "position_tier", "impression_tier"]

for c in numeric_feats:
    if c in df.columns:
        df[c] = pd.to_numeric(df[c], errors="coerce").replace([np.inf, -np.inf], np.nan).fillna(0)
    else:
        df[c] = 0

for c in cat_feats:
    if c not in df.columns:
        df[c] = "unknown"
    df[c] = df[c].fillna("unknown").astype(str)

X_num = df[numeric_feats]
X_cat = pd.get_dummies(df[cat_feats], prefix=cat_feats, dtype=float)
X = pd.concat([X_num.reset_index(drop=True), X_cat.reset_index(drop=True)], axis=1)
y = df["is_declining_label"].astype(int)
clients = df["client_id"]

# Client-holdout split
unique_clients = clients.unique()
train_clients, test_clients = train_test_split(
    unique_clients, test_size=0.2, random_state=RANDOM_STATE
)
train_mask = clients.isin(train_clients)
test_mask = clients.isin(test_clients)

X_train, X_test = X.loc[train_mask], X.loc[test_mask]
y_train, y_test = y.loc[train_mask], y.loc[test_mask]
df_test = df.loc[test_mask].copy()

print("Train rows:", len(X_train), "| Test rows:", len(X_test))
print("Train clients:", len(train_clients), "| Test clients:", len(test_clients))
print("Train base rate:", round(y_train.mean(), 3))
print("Test base rate:", round(y_test.mean(), 3))
print("Feature count:", X.shape[1])
print("Leak check — trend cols in X?", any("trend" in c.lower() for c in X.columns))

Train rows: 26581 | Test rows: 3419
Train clients: 25 | Test clients: 7
Train base rate: 0.544
Test base rate: 0.524
Feature count: 36
Leak check — trend cols in X? False


## 3. Train + compare vs my baseline

*Same data, same metric, same split as your Week-4 baseline. Show the table.*

Same test clients, same Precision@K. Baseline score is the transparent rule from ML-07 (visibility + freshness + position + depth). Models are ranked by predicted probability of decline.

In [8]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

def precision_at_k(y_true, scores, k):
    y_true = np.asarray(y_true)
    scores = np.asarray(scores)
    order = np.argsort(-scores)
    k = min(k, len(order))
    return float(y_true[order[:k]].mean()) if k else 0.0

# --- Hand baseline on the TEST set (same spirit as ML-07) ---
def percentile_rank(s):
    return s.rank(pct=True, method="average").fillna(0)

def normalize(s):
    s = s.astype(float)
    lo, hi = s.min(), s.max()
    return (s - lo) / (hi - lo) if hi != lo else pd.Series(0.0, index=s.index)

bt = df_test.copy()
bt["visibility_score"] = percentile_rank(np.log1p(bt["impressions_90d"]))
bt["freshness_risk_score"] = percentile_rank(bt["days_since_last_update"])
bt["position_opportunity_score"] = (
    (1 - normalize(bt["avg_position"].clip(lower=1, upper=50)))
    * bt["visibility_score"]
    * (bt["avg_position"] > 0).astype(int)
)
bt["depth_gap_score"] = (1 - percentile_rank(bt["word_count"].fillna(0))) * bt["visibility_score"]
bt["baseline_score"] = (
    0.40 * bt["visibility_score"]
    + 0.30 * bt["freshness_risk_score"]
    + 0.25 * bt["position_opportunity_score"]
    + 0.05 * bt["depth_gap_score"]
).clip(0, 1)

# --- Models ---
log_reg = Pipeline([
    ("scaler", StandardScaler()),
    ("model", LogisticRegression(max_iter=1000, random_state=RANDOM_STATE)),
])
rf = RandomForestClassifier(
    n_estimators=200, max_depth=12, min_samples_leaf=5,
    random_state=RANDOM_STATE, n_jobs=-1
)

log_reg.fit(X_train, y_train)
rf.fit(X_train, y_train)

proba_lr = log_reg.predict_proba(X_test)[:, 1]
proba_rf = rf.predict_proba(X_test)[:, 1]
base_scores = bt["baseline_score"].values
y_te = y_test.values

rows = []
for name, scores in [
    ("Base rate (random)", np.full(len(y_te), y_te.mean())),
    ("Hand baseline", base_scores),
    ("Logistic Regression", proba_lr),
    ("Random Forest", proba_rf),
]:
    rows.append({
        "method": name,
        "Precision@20": round(precision_at_k(y_te, scores, 20), 3),
        "Precision@50": round(precision_at_k(y_te, scores, 50), 3),
        "Precision@100": round(precision_at_k(y_te, scores, 100), 3),
        "test_base_rate": round(y_te.mean(), 3),
    })

results = pd.DataFrame(rows)
display(results)

print("\nLift RF vs hand baseline @50:",
      round(results.loc[results.method=="Random Forest", "Precision@50"].values[0] /
            max(results.loc[results.method=="Hand baseline", "Precision@50"].values[0], 1e-6), 2), "x")

,method,Precision@20,Precision@50,Precision@100,test_base_rate
0,Base rate (random),0.65,0.62,0.59,0.524
1,Hand baseline,0.30,0.42,0.50,0.524
2,Logistic Regression,0.95,0.86,0.81,0.524
3,Random Forest,0.80,0.76,0.78,0.524



Lift RF vs hand baseline @50: 1.81 x


## 4. Errors and interpretation

*Where is the model wrong? What does it lean on? A short error analysis beats a big metric table.*

**Short error analysis: what the model leans on, and where it is still wrong.**

In [9]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

# Feature importance (RF)
imp = pd.Series(rf.feature_importances_, index=X.columns).sort_values(ascending=False).head(12)
print("Top RF features:")
print(imp.round(4).to_string())
print()

# Error slice: top-50 RF picks that are NOT declining
df_test = df_test.copy()
df_test["rf_proba"] = proba_rf
top50 = df_test.nlargest(50, "rf_proba")
false_pos = top50[top50["is_declining_label"] == 0]
print("False positives in RF top-50:", len(false_pos), "/", 50)
if len(false_pos):
    print(false_pos[["content_id", "impressions_90d", "avg_position", "ctr",
                     "days_since_last_update", "word_count", "trend_direction", "rf_proba"]].head(8).to_string(index=False))

print("\nInterpretation notes go in the markdown cell above.")


Top RF features:
impressions_90d           0.1708
avg_position              0.1319
content_age_days          0.1288
word_count                0.0825
ctr                       0.0433
scroll_rate               0.0421
clicks_90d                0.0388
days_since_last_update    0.0352
pageviews_90d             0.0312
users_90d                 0.0283
sessions_90d              0.0278
position_tier_top_3       0.0241

False positives in RF top-50: 12 / 50
          content_id  impressions_90d  avg_position  ctr  days_since_last_update  word_count trend_direction  rf_proba
content_0b47dae0c7f9             1191          23.1 0.00                     103      1514.0          stable  0.840705
content_3164f3076003             2696          16.1 0.04                     104      1274.0              up  0.837109
content_846bb4dd8b44              870          17.6 0.11                     104      1492.0          stable  0.827226
content_197a5b1ed096              605          14.4 0.00                

**What it leans on (directional):**

High importance usually lands on volume/visibility, position, age/update recency, and CTR-like rates — signals a human editor already watches. That is reassuring (not a single mysterious feature dominating).

**Where it is wrong:**

False positives in the top-50 are often pages that still look “important” (high impressions or aging) but are stable or up. The model over-weights “looks worth a review” and under-weights the pure trend signal (which we correctly refused to leak in).

**Takeaway:**

The learned score improves Precision@50 over the hand rule on this client-holdout test set, but it is still decision-support, not a guarantee. An editor should treat the ranked list as a starting queue and apply judgment on seasonal or intentional flat pages.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.